# 📓 Day 3: Grounded Generation & Citation Formatting
## VERA (Verified Evidence Retrieval Assistant)

**Objective**: Generate strict evidence-grounded recommendations with explicit in-line citations and test refusal behavior.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.embeddings.embedder import MedicalEmbedder
from src.embeddings.vector_store import VectorStoreManager
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.generator import ClinicalGenerator
from src.generation.citation_formatter import CitationFormatter
from src.utils.helpers import load_json

print("Imports loaded!")

Imports loaded!


### 1. Initialize Retriever & Generator

In [2]:
chunks_data = load_json("../data/processed/chunk_catalog.json")
embedder = MedicalEmbedder()
vector_store = VectorStoreManager(persist_dir="../data/vector_db", embedder=embedder)
retriever = HybridRetriever(vector_store, all_chunks=chunks_data)

# Initializing generator (will use OpenAI if API key set, or deterministic fallback)
generator = ClinicalGenerator(provider="openai", model_name="gpt-4o-mini", temperature=0.0)
print("Generator initialized!")

2026-08-16 21:54:23 | INFO     | src.embeddings.embedder:59 - Loading Local Model: 'BAAI/bge-small-en-v1.5' on device 'cpu'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-16 21:54:28 | SUCCESS  | src.embeddings.embedder:61 - Model 'BAAI/bge-small-en-v1.5' loaded successfully (dim=384)


d:\AI Hackathon\New data\src\embeddings\embedder.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.success(f"Model '{self.model_name}' loaded successfully (dim={self.model.get_sentence_embedding_dimension()})")


2026-08-16 21:54:29 | INFO     | src.embeddings.vector_store:44 - VectorStoreManager connected to ChromaDB collection: 'vera_clinical_guidelines' (Current count: 188)
2026-08-16 21:54:29 | INFO     | src.retrieval.hybrid_retriever:34 - Initialized BM25 index with 94 documents.
2026-08-16 21:54:30 | WARNING  | src.generation.generator:60 - OPENAI_API_KEY not found in environment.
Generator initialized!


### 2. Generate Grounded Response for In-Scope Query

In [3]:
query = "What are the three FDA-approved disease-modifying therapies for Spinal Muscular Atrophy?"
retrieved_chunks = retriever.retrieve(query, top_k=3)

response = generator.generate_response(query, retrieved_chunks)
print("=== GENERATED CLINICAL RESPONSE ===")
print(response["answer"])

print("\n=== EXTRACTED CITATIONS ===")
for c in response["citations"]:
    print(f"- Doc: {c['doc_name']} | Section: {c['section']} | Page: {c['page']}")

2026-08-16 21:54:30 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 3 chunks.
=== GENERATED CLINICAL RESPONSE ===
### Clinical Recommendation / Direct Answer:
Based on the retrieved clinical evidence for query 'What are the three FDA-approved disease-modifying therapies for Spinal Muscular Atrophy?', recommendations indicate: RESEARCHARTICLE OPENACCESS Spinal Muscular Atrophy Update in Best Practices RecommendationsforTreatmentConsiderations MaryK.Schroth,MD,JenniferDeans,MHA,MS,CCLS,DianaX.BharuchaGoebel,MD,W.BryanBurnette,MD,MS, Correspondence BasilT.Darras,MD,BakriH.Elsheikh,MBBS,FRCP(Edin),MarciaV... [ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf | Section: General Overview | Page: 1]

### Supporting Evidence & Excerpts:
- "RESEARCHARTICLE OPENACCESS Spinal Muscular Atrophy Update in Best Practices RecommendationsforTreatmentConsiderations MaryK.Schroth,MD,JenniferDeans,MHA,MS,CCLS,DianaX.BharuchaGoebel,MD,W.BryanBurnette,MD,MS, Corresp

### 3. Verify Citation Validity Against Retrieved Context

In [4]:
is_valid, accuracy = CitationFormatter.validate_citations_against_context(
    response["answer"], retrieved_chunks
)
print(f"Citations Valid: {is_valid} (Citation Precision: {accuracy * 100:.1f}%)")

Citations Valid: True (Citation Precision: 100.0%)
